# Obstacle Detection - Custom YOLOv8n Training

This notebook trains a YOLOv8n model for real-time obstacle detection on the
AgroBot rover. The model detects obstacles that the rover must avoid during
autonomous navigation.

**Hardware requirements:** Google Colab with T4 GPU runtime.

**Classes:** person, vehicle, animal, rock, stump, fence, ditch

**Expected output:** `obstacle_model_quant_edgetpu.tflite` for deployment
to `models/` on the Raspberry Pi.

**Integration:** Loaded by `pi/ai/obstacle_detection.py` via the
`ObstacleDetector` class.

In [ ]:
# Install dependencies (pinned versions for reproducibility)
!pip install -q ultralytics==8.1.0 onnx==1.15.0 onnxruntime==1.17.0 matplotlib==3.8.2

In [ ]:
import os
import yaml
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print(f"Working directory: {os.getcwd()}")

## Custom Dataset Structure

The obstacle detection dataset uses standard YOLO annotation format:

```
dataset/
  train/
    images/
      img_001.jpg
      img_002.jpg
    labels/
      img_001.txt
      img_002.txt
  valid/
    images/
    labels/
  test/
    images/
    labels/
  dataset.yaml
```

**YOLO annotation format** (one line per object in the `.txt` file):
```
<class_id> <x_center> <y_center> <width> <height>
```

All coordinates are normalized to [0, 1] relative to image dimensions.

**Example annotation (img_001.txt):**
```
0 0.45 0.60 0.12 0.30
3 0.72 0.80 0.08 0.06
```
This means: a `person` (class 0) centered at (45%, 60%) with 12% width and
30% height, and a `rock` (class 3) centered at (72%, 80%).

In [ ]:
# Sample dataset.yaml for obstacle classes
DATASET_DIR = '/content/obstacle_dataset'

dataset_yaml = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 7,
    'names': [
        'person',
        'vehicle',
        'animal',
        'rock',
        'stump',
        'fence',
        'ditch',
    ],
}

# Create dataset directory structure
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(DATASET_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, split, 'labels'), exist_ok=True)

yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Dataset config saved to: {yaml_path}")
print(f"\nClasses ({dataset_yaml['nc']}):")
for i, name in enumerate(dataset_yaml['names']):
    print(f"  {i}: {name}")

print("\nPopulate the dataset directories with your annotated images.")
print("Recommended: at least 500 images per class for good performance.")
print("Sources: custom field captures, COCO subset, Open Images subset.")

## Data Augmentation Strategy

YOLOv8 applies augmentation during training. Key augmentation parameters
tuned for outdoor agricultural environments:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `mosaic` | 1.0 | Combines 4 images to improve small object detection |
| `mixup` | 0.15 | Blends images to improve generalization |
| `hsv_h` | 0.015 | Handles varying daylight hue shifts |
| `hsv_s` | 0.7 | Handles shadow/sun saturation variation |
| `hsv_v` | 0.4 | Handles brightness changes (shade to direct sun) |
| `degrees` | 15.0 | Rover camera tilt on uneven terrain |
| `translate` | 0.1 | Object position variation |
| `scale` | 0.5 | Objects at varying distances |
| `fliplr` | 0.5 | Horizontal symmetry |
| `flipud` | 0.1 | Minor vertical flip for ground objects |

Additional offline augmentation can be applied using Albumentations
for extreme weather conditions (rain, fog, dust).

In [ ]:
# YOLOv8n training from COCO pretrained weights
model = YOLO('yolov8n.pt')  # Start from COCO pretrained weights

print(f"Model: YOLOv8n")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"\nStarting training...")

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    # Augmentation parameters optimized for outdoor obstacles
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    # Training hyperparameters
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    # Output settings
    project='obstacle_detection',
    name='yolov8n_obstacles',
    save=True,
    plots=True,
)

print(f"\nTraining complete!")
print(f"Best weights: {results.save_dir}/weights/best.pt")

In [ ]:
# Evaluation with mAP and per-class AP
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
val_results = best_model.val(data=yaml_path, imgsz=640, device=0)

print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"  mAP@0.5:      {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")

# Per-class performance
print(f"\nPer-class AP@0.5:")
print(f"{'Class':<15} {'AP50':>8} {'AP50-95':>10}")
print("-" * 35)
for i, name in enumerate(dataset_yaml['names']):
    if i < len(val_results.box.ap50):
        ap50 = val_results.box.ap50[i]
        ap = val_results.box.ap[i] if i < len(val_results.box.ap) else 0
        print(f"{name:<15} {ap50:>8.4f} {ap:>10.4f}")

# Display training results
from IPython.display import Image, display
display(Image(filename=f'{results.save_dir}/results.png', width=800))
display(Image(filename=f'{results.save_dir}/confusion_matrix_normalized.png', width=600))

In [ ]:
# Export to TFLite + ONNX
best_model = YOLO(f'{results.save_dir}/weights/best.pt')

# Export to ONNX (for flexibility and edge deployment alternatives)
onnx_path = best_model.export(
    format='onnx',
    imgsz=320,
    simplify=True,
    dynamic=False,
)
print(f"ONNX model exported to: {onnx_path}")
print(f"ONNX size: {os.path.getsize(onnx_path) / 1024 / 1024:.2f} MB")

# Export to TFLite with INT8 quantization for Edge TPU
tflite_path = best_model.export(
    format='tflite',
    imgsz=320,
    int8=True,
)
print(f"\nTFLite model exported to: {tflite_path}")
print(f"TFLite size: {os.path.getsize(tflite_path) / 1024 / 1024:.2f} MB")

In [ ]:
# Edge TPU compilation
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update && sudo apt-get install -y edgetpu-compiler

# Compile for Edge TPU
os.makedirs('output', exist_ok=True)
!edgetpu_compiler -s -o output/ {tflite_path}

print("\nEdge TPU compilation complete!")
!ls -la output/

In [ ]:
# Real-time inference benchmark
# Measure inference speed to ensure it meets the rover's real-time requirements
# Target: >= 15 FPS on Coral Edge TPU for safe obstacle avoidance

import numpy as np

# Benchmark with the PyTorch model (GPU)
dummy_image = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(10):
    best_model.predict(dummy_image, verbose=False)

# Timed inference
num_iterations = 100
start_time = time.time()
for _ in range(num_iterations):
    best_model.predict(dummy_image, verbose=False)
elapsed = time.time() - start_time

fps = num_iterations / elapsed
latency_ms = (elapsed / num_iterations) * 1000

print("=" * 50)
print("INFERENCE BENCHMARK (PyTorch, GPU)")
print("=" * 50)
print(f"  Iterations: {num_iterations}")
print(f"  Total time: {elapsed:.2f}s")
print(f"  FPS:        {fps:.1f}")
print(f"  Latency:    {latency_ms:.1f}ms")
print(f"\nNote: On Coral Edge TPU expect ~25-30 FPS with 320x320 input.")
print(f"This exceeds the 15 FPS minimum for obstacle avoidance.")

## Integration with pi/ai/obstacle_detection.py

The exported Edge TPU model integrates with the AgroBot obstacle detection
module:

```python
# In pi/ai/obstacle_detection.py
from ai.tflite_backend import TFLiteBackend

class ObstacleDetector:
    def __init__(self, model_path='models/obstacle_model_quant_edgetpu.tflite'):
        self.backend = TFLiteBackend(model_path, use_edgetpu=True)
        self.classes = ['person', 'vehicle', 'animal', 'rock',
                        'stump', 'fence', 'ditch']
        self.conf_threshold = 0.4

    def detect(self, frame):
        # Preprocess: resize to 320x320, normalize
        input_tensor = self._preprocess(frame)
        # Run inference
        outputs = self.backend.invoke(input_tensor)
        # Postprocess: NMS, filter by confidence
        detections = self._postprocess(outputs)
        return detections
```

### Deployment steps:
1. Download `obstacle_model_quant_edgetpu.tflite` from this notebook
2. Place in `models/` directory on the Raspberry Pi
3. The `ObstacleDetector` loads the model at startup
4. The mission planner queries detections for path replanning

### Safety thresholds:
- `person` detection triggers immediate stop (conf > 0.3)
- `vehicle` detection triggers slow-down and avoidance (conf > 0.4)
- Other obstacles trigger path replanning (conf > 0.5)

In [ ]:
# Download models for deployment
from google.colab import files

# Find the Edge TPU compiled model
edgetpu_models = [f for f in os.listdir('output') if 'edgetpu' in f]
if edgetpu_models:
    edgetpu_model = edgetpu_models[0]
    files.download(f'output/{edgetpu_model}')
    print(f"Downloaded: output/{edgetpu_model}")

# Also download ONNX model (useful for debugging on desktop)
if os.path.exists(onnx_path):
    files.download(onnx_path)
    print(f"Downloaded: {onnx_path}")

print("\nDeployment instructions:")
print("  1. Rename Edge TPU model to: obstacle_model_quant_edgetpu.tflite")
print("  2. Copy to: models/obstacle_model_quant_edgetpu.tflite")
print("  3. Restart the agrobot-pipeline service")
print("  4. Verify with: python3 -c 'from ai.obstacle_detection import ObstacleDetector'")